<a href="https://colab.research.google.com/github/AnnesaAD05/Python-Projects-Informatics-1st-year-annesa/blob/main/%F0%9F%8E%AC_Movie_Night_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import requests
import os
from google.colab import userdata

# Retrieve TMDB API key from Colab secrets
TMDB_API_KEY = userdata.get('TMDB_API_KEY')
TMDB_BASE_URL = 'https://api.themoviedb.org/3'

# Fetch genre list to map genre names to IDs and vice-versa
def get_genres():
    url = f"{TMDB_BASE_URL}/genre/movie/list"
    params = {'api_key': TMDB_API_KEY}
    response = requests.get(url, params=params)
    response.raise_for_status()
    genres = response.json().get('genres', [])

    genre_map = {genre['id']: genre['name'] for genre in genres}
    reverse_genre_map = {genre['name'].lower(): genre['id'] for genre in genres}
    # Add common aliases
    reverse_genre_map['sci-fi'] = reverse_genre_map.get('science fiction')
    return genre_map, reverse_genre_map

genre_id_to_name, genre_name_to_id = get_genres()

print("TMDB API setup complete and genres fetched.")

TMDB API setup complete and genres fetched.


In [7]:
def search_movies(genre_input, year_gte, num_results=3):
    """
    Searches for movies on TMDB based on genre and release year.

    Args:
        genre_input (str): The name of the genre (e.g., 'Sci-Fi', 'Action').
        year_gte (int): Minimum release year (e.g., 2020).
        num_results (int): Number of top results to return.

    Returns:
        list: A list of dictionaries, each representing a movie.
    """
    genre_id = genre_name_to_id.get(genre_input.lower())
    if not genre_id:
        print(f"Error: Genre '{genre_input}' not found. Available genres are: {', '.join(genre_name_to_id.keys())}")
        return []

    url = f"{TMDB_BASE_URL}/discover/movie"
    params = {
        'api_key': TMDB_API_KEY,
        'with_genres': genre_id,
        'primary_release_date.gte': f'{year_gte}-01-01',
        'sort_by': 'popularity.desc', # Sort by popularity to get good recommendations
        'language': 'en-US' # Ensure results are in English
    }

    response = requests.get(url, params=params)
    response.raise_for_status() # Raise an exception for bad status codes
    data = response.json()

    movies = []
    for result in data.get('results', [])[:num_results]:
        movie_genres = [genre_id_to_name.get(gid, 'Unknown') for gid in result.get('genre_ids', [])]
        movies.append({
            'title': result.get('title'),
            'rating': result.get('vote_average'),
            'release_year': result.get('release_date', '')[:4], # Extract year from date
            'genres': ' / '.join(movie_genres),
            'overview': result.get('overview')
        })
    return movies

print("Movie search function defined.")

Movie search function defined.


In [8]:
# User enters:
# Genre: Sci-Fi
# Year: 2020+

genre_selection = "Sci-Fi"
year_selection = 2020

print("🎬 Movie Night\n")
print(f"Searching for movies in Genre: {genre_selection}, Year: {year_selection}+\n")

recommended_movies = search_movies(genre_selection, year_selection, num_results=3)

if recommended_movies:
    for movie in recommended_movies:
        print(f"{movie['title']}")
        print(f"⭐ Rating: {movie['rating']}/10")
        print(f"📅 {movie['release_year']}")
        print(f"🎭 {movie['genres']}")
        print(f"Overview: {movie['overview']}\n")
else:
    print("No movies found matching your criteria.")

🎬 Movie Night

Searching for movies in Genre: Sci-Fi, Year: 2020+

Spider-Man: Brand New Day
⭐ Rating: 7.865/10
📅 2026
🎭 Science Fiction / Action / Adventure
Overview: Fighting crime full-time as Spider-Man in a world that doesn't remember him—and the pressure of seeing his old friends move on without him—sparks a change in Peter Parker he may not have the power to control. But that transformation might also be the only thing that can stop a shocking new threat to the city and those he loves - a powerful villain no one can even see.

Resident Evil
⭐ Rating: 7.3/10
📅 2026
🎭 Horror / Science Fiction / Adventure
Overview: Medical courier Bryan unwittingly finds himself fighting for survival as one fateful, horrifying night collapses around him in chaos.

The End of Oak Street
⭐ Rating: 7.1/10
📅 2026
🎭 Science Fiction / Mystery / Thriller
Overview: After a mysterious cosmic event rips Oak Street from suburbia and transports their neighborhood to someplace unknown, the Platt family soon dis